In [2]:
pip install -U jupyterlab_widgets

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import GPT2LMHeadModel, GPT2Config

torch.manual_seed(1337)

In [2]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

print(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [3]:
config = model.config

print(config)

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "use_cache": true,
  "vocab_size": 50257
}



In [4]:
def get_device(): 
        if torch.cuda.is_available(): 
            return torch.device("cuda") 
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps") 
        return torch.device("cpu")

device = get_device()

In [5]:
print("device:", device)

device: mps


In [6]:
print("Vocabulary size :", config.vocab_size)
print("Context length  :", config.n_positions)
print("Embedding dim   :", config.n_embd)
print("Layers          :", config.n_layer)
print("Attention heads :", config.n_head)

Vocabulary size : 50257
Context length  : 1024
Embedding dim   : 768
Layers          : 12
Attention heads : 12


In [7]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {num_params:,}")
print(f"Total parameters: {num_params / 1e6:.2f}M")

Total parameters: 124,439,808
Total parameters: 124.44M


In [8]:
for name, module in model.named_modules():
    if name:
        print(name, "->", module.__class__.__name__)

transformer -> GPT2Model
transformer.wte -> Embedding
transformer.wpe -> Embedding
transformer.drop -> Dropout
transformer.h -> ModuleList
transformer.h.0 -> GPT2Block
transformer.h.0.ln_1 -> LayerNorm
transformer.h.0.attn -> GPT2Attention
transformer.h.0.attn.c_attn -> Conv1D
transformer.h.0.attn.c_proj -> Conv1D
transformer.h.0.attn.attn_dropout -> Dropout
transformer.h.0.attn.resid_dropout -> Dropout
transformer.h.0.ln_2 -> LayerNorm
transformer.h.0.mlp -> GPT2MLP
transformer.h.0.mlp.c_fc -> Conv1D
transformer.h.0.mlp.c_proj -> Conv1D
transformer.h.0.mlp.act -> NewGELUActivation
transformer.h.0.mlp.dropout -> Dropout
transformer.h.1 -> GPT2Block
transformer.h.1.ln_1 -> LayerNorm
transformer.h.1.attn -> GPT2Attention
transformer.h.1.attn.c_attn -> Conv1D
transformer.h.1.attn.c_proj -> Conv1D
transformer.h.1.attn.attn_dropout -> Dropout
transformer.h.1.attn.resid_dropout -> Dropout
transformer.h.1.ln_2 -> LayerNorm
transformer.h.1.mlp -> GPT2MLP
transformer.h.1.mlp.c_fc -> Conv1D
tran

In [9]:
block = model.transformer.h[0]

print(block)

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [10]:
print("Token embedding:", model.transformer.wte.weight.shape)
print("Position embedding:", model.transformer.wpe.weight.shape)

print("Attention QKV:", block.attn.c_attn.weight.shape)
print("Attention output:", block.attn.c_proj.weight.shape)

print("MLP first layer:", block.mlp.c_fc.weight.shape)
print("MLP second layer:", block.mlp.c_proj.weight.shape)

Token embedding: torch.Size([50257, 768])
Position embedding: torch.Size([1024, 768])
Attention QKV: torch.Size([768, 2304])
Attention output: torch.Size([768, 768])
MLP first layer: torch.Size([768, 3072])
MLP second layer: torch.Size([3072, 768])


In [11]:
torch.manual_seed(1337)

reference_model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2"
)

config = reference_model.config

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [12]:
print(reference_model.transformer.wte)
print(reference_model.transformer.wpe)

print("token embeddings :", reference_model.transformer.wte.weight.shape)
print("position embeddings:", reference_model.transformer.wpe.weight.shape)

Embedding(50257, 768)
Embedding(1024, 768)
token embeddings : torch.Size([50257, 768])
position embeddings: torch.Size([1024, 768])


In [13]:
class GPT2Embeddings(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        tok_emb = self.wte(idx)
        pos_emb = self.wpe(positions)

        return tok_emb + pos_emb

In [14]:
embeddings = GPT2Embeddings(
    config.vocab_size,
    config.n_positions,
    config.n_embd,
)

idx = torch.randint(
    0,
    config.vocab_size,
    (2, 8),
)

x = embeddings(idx)

print("input :", idx.shape)
print("output:", x.shape)

input : torch.Size([2, 8])
output: torch.Size([2, 8, 768])


In [15]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True):
        super().__init__()

        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5,
        )

In [16]:
ln = LayerNorm(config.n_embd)

x = torch.randn(2, 8, config.n_embd)
y = ln(x)

print("input :", x.shape)
print("output:", y.shape)
print("mean  :", y.mean().item())
print("std   :", y.std().item())

input : torch.Size([2, 8, 768])
output: torch.Size([2, 8, 768])
mean  : 0.0
std   : 1.0000356435775757


In [17]:
import math

In [18]:

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head

        # GPT-2 combines Q, K and V projections.
        self.c_attn = nn.Linear(
            n_embd,
            3 * n_embd
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Causal mask.
        mask = torch.tril(
            torch.ones(block_size, block_size)
        )

        self.register_buffer(
            "bias",
            mask.view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # [B, T, C] -> [B, n_head, T, head_dim]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Attention scores.
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # preventing attending to future tokens.
        att = att.masked_fill(
            self.bias[:, :, :T, :T] == 0,
            float("-inf")
        )

        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # weighted combination of values.
        y = att @ v

        # [B, n_head, T, head_dim]
        # -> [B, T, n_head, head_dim]
        y = y.transpose(1, 2).contiguous()

        # merge heads.
        y = y.view(B, T, C)

        # output projection.
        y = self.c_proj(y)
        y = self.resid_dropout(y)

        return y
    

In [19]:
attention = CausalSelfAttention(
    n_embd=config.n_embd,
    n_head=config.n_head,
    block_size=config.n_positions,
)

x = torch.randn(2, 8, config.n_embd)

y = attention(x)
with torch.no_grad():
     qkv = attention.c_attn(x)
     print("QKV:", qkv.shape)
print("input :", x.shape)
print("output:", y.shape)

QKV: torch.Size([2, 8, 2304])
input : torch.Size([2, 8, 768])
output: torch.Size([2, 8, 768])


## GPT 2 MLP

In [20]:
class GPT2MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()

        self.c_fc = nn.Linear(
            n_embd,
            4 * n_embd
        )

        self.c_proj = nn.Linear(
            4 * n_embd,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)

        return x

In [21]:
mlp = GPT2MLP(config.n_embd)

x = torch.randn(2, 8, config.n_embd)
y = mlp(x)

print(y.shape)

torch.Size([2, 8, 768])


In [22]:
class GPT2Block(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        block_size,
        dropout=0.0,
    ):
        super().__init__()

        self.ln_1 = LayerNorm(n_embd)

        self.attn = CausalSelfAttention(
            n_embd,
            n_head,
            block_size,
            dropout,
        )

        self.ln_2 = LayerNorm(n_embd)

        self.mlp = GPT2MLP(
            n_embd,
            dropout,
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))

        return x

In [23]:
block = GPT2Block(
    config.n_embd,
    config.n_head,
    config.n_positions,
)

x = torch.randn(2, 8, config.n_embd)
y = block(x)

print(y.shape)

torch.Size([2, 8, 768])


In [24]:
for name, param in block.named_parameters():
    print(f"{name:30s} {tuple(param.shape)}")

ln_1.weight                    (768,)
ln_1.bias                      (768,)
attn.c_attn.weight             (2304, 768)
attn.c_attn.bias               (2304,)
attn.c_proj.weight             (768, 768)
attn.c_proj.bias               (768,)
ln_2.weight                    (768,)
ln_2.bias                      (768,)
mlp.c_fc.weight                (3072, 768)
mlp.c_fc.bias                  (3072,)
mlp.c_proj.weight              (768, 3072)
mlp.c_proj.bias                (768,)


## Weight-tied language model head

In [25]:
class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        n_layer,
        n_head,
        n_embd,
        dropout=0.0,
    ):
        super().__init__()

        self.block_size = block_size

        self.transformer = nn.ModuleDict({
            "wte": nn.Embedding(vocab_size, n_embd),
            "wpe": nn.Embedding(block_size, n_embd),
            "h": nn.ModuleList([
                GPT2Block(
                    n_embd=n_embd,
                    n_head=n_head,
                    block_size=block_size,
                    dropout=dropout,
                )
                for _ in range(n_layer)
            ]),
            "ln_f": LayerNorm(n_embd),
        })

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False,
        )

        # tying token embeddings and output projection weights.
        self.lm_head.weight = self.transformer["wte"].weight

    def forward(self, idx, targets=None):
        B, T = idx.shape

        assert T <= self.block_size, (
            f"sequence length {T} exceeds block size "
            f"{self.block_size}"
        )

        tok_emb = self.transformer["wte"](idx)

        pos = torch.arange(
            T,
            device=idx.device,
        )

        pos_emb = self.transformer["wpe"](pos)

        x = tok_emb + pos_emb

        for block in self.transformer["h"]:
            x = block(x)

        x = self.transformer["ln_f"](x)

        logits = self.lm_head(x)
        
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )
        else:
            loss = None
            
        return logits, loss


    @torch.no_grad()
    def generate(self, idx, max_new_tokens, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
    
            # logits = self(idx_cond)
            logits, _ = self(idx_cond)
    
            logits = logits[:, -1, :]
            
            probs = F.softmax(logits, dim=-1)
            
            topk_probs, topk_indices = torch.topk(
                probs,
                top_k,
                dim=-1,
            )

            ix = torch.multinomial(
                topk_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                topk_indices,
                -1,
                ix,
            )

            idx = torch.cat(
                (idx, next_token),
                dim=1,
            )
    
        return idx

In [26]:
model = GPT2(
    vocab_size=config.vocab_size,
    block_size=config.n_positions,
    n_layer=config.n_layer,
    n_head=config.n_head,
    n_embd=config.n_embd,
)

idx = torch.randint(
    0,
    config.vocab_size,
    (2, 16),
)

logits, loss = model(idx)

print("input :", idx.shape)
print("logits:", logits.shape)

input : torch.Size([2, 16])
logits: torch.Size([2, 16, 50257])


In [27]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"parameters: {num_params:,}")
print(f"parameters: {num_params / 1e6:.2f}M")

parameters: 124,439,808
parameters: 124.44M


In [28]:
print(
    model.transformer["wte"].weight
    is model.lm_head.weight
)

True


In [29]:
print(model)

GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): GPT2MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [30]:
print(reference_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [31]:
from transformers import  GPT2TokenizerFast
tokenizer = GPT2TokenizerFast.from_pretrained(
    "openai-community/gpt2"
)
print("vocab size:", tokenizer.vocab_size)



vocab size: 50257


In [32]:
print("model vocab size:", config.vocab_size)
print("tokenizer vocab size:", tokenizer.vocab_size)


text = "Hello, my name is Abhijeet."

tokens = tokenizer.encode(text)

print(tokens)


decoded = tokenizer.decode(tokens)

print(decoded)


for token_id in tokens:
    token_bytes = tokenizer.decode([token_id])
    print(token_id, repr(token_bytes))
    
    
    
    
tokens = tokenizer.encode(text)

idx = torch.tensor(
    [tokens],
    dtype=torch.long,
)

print(idx.shape)
print(idx)

model vocab size: 50257
tokenizer vocab size: 50257
[15496, 11, 616, 1438, 318, 2275, 71, 2926, 68, 316, 13]
Hello, my name is Abhijeet.
15496 'Hello'
11 ','
616 ' my'
1438 ' name'
318 ' is'
2275 ' Ab'
71 'h'
2926 'ij'
68 'e'
316 'et'
13 '.'
torch.Size([1, 11])
tensor([[15496,    11,   616,  1438,   318,  2275,    71,  2926,    68,   316,
            13]])


In [33]:
with torch.no_grad():
    logits, loss = model(idx)

print(logits.shape)


torch.Size([1, 11, 50257])


In [34]:
def load_gpt2_weights(model, reference_model):
    with torch.no_grad():

        # Token and positional embeddings
        model.transformer["wte"].weight.copy_(
            reference_model.transformer.wte.weight
        )

        model.transformer["wpe"].weight.copy_(
            reference_model.transformer.wpe.weight
        )

        # Transformer BLOCKS
        for i, block in enumerate(model.transformer["h"]):

            ref_block = reference_model.transformer.h[i]

            # LayerNorm 1
            block.ln_1.weight.copy_(
                ref_block.ln_1.weight
            )

            block.ln_1.bias.copy_(
                ref_block.ln_1.bias
            )

            # attention QKV
            block.attn.c_attn.weight.copy_(
                ref_block.attn.c_attn.weight.T
            )

            block.attn.c_attn.bias.copy_(
                ref_block.attn.c_attn.bias
            )

            # attention output projection
            block.attn.c_proj.weight.copy_(
                ref_block.attn.c_proj.weight.T
            )

            block.attn.c_proj.bias.copy_(
                ref_block.attn.c_proj.bias
            )

            # LayerNorm 2
            block.ln_2.weight.copy_(
                ref_block.ln_2.weight
            )

            block.ln_2.bias.copy_(
                ref_block.ln_2.bias
            )

            # MLP input projection
            block.mlp.c_fc.weight.copy_(
                ref_block.mlp.c_fc.weight.T
            )

            block.mlp.c_fc.bias.copy_(
                ref_block.mlp.c_fc.bias
            )

            # MLP output projection
            block.mlp.c_proj.weight.copy_(
                ref_block.mlp.c_proj.weight.T
            )

            block.mlp.c_proj.bias.copy_(
                ref_block.mlp.c_proj.bias
            )

        # last LayerNorm
        model.transformer["ln_f"].weight.copy_(
            reference_model.transformer.ln_f.weight
        )

        model.transformer["ln_f"].bias.copy_(
            reference_model.transformer.ln_f.bias
        )

    return model

In [35]:
model = GPT2(
    vocab_size=config.vocab_size,
    block_size=config.n_positions,
    n_layer=config.n_layer,
    n_head=config.n_head,
    n_embd=config.n_embd,
)

In [36]:
load_gpt2_weights(
    model,
    reference_model,
)

GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): GPT2MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [37]:
print(
    torch.equal(
        model.transformer["wte"].weight,
        reference_model.transformer.wte.weight,
    )
)

True


In [38]:
print(
    torch.equal(
        model.transformer["ln_f"].weight,
        reference_model.transformer.ln_f.weight,
    )
)

True


In [39]:
print(
    torch.equal(
        model.transformer["h"][0].attn.c_attn.weight,
        reference_model.transformer.h[0].attn.c_attn.weight.T,
    )
)

True


In [40]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"{num_params:,}")
print(f"{num_params / 1e6:.2f}M")

124,439,808
124.44M


In [41]:
text = "The quick brown fox jumps over the lazy dog."

tokens = tokenizer.encode(text)

idx = torch.tensor(
    [tokens],
    dtype=torch.long,
)

In [43]:
reference_model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2"
)
reference_model.eval()

model = model.to(device)
model.eval()

idx_cpu = idx
idx_device = idx.to(device)

print("reference device:", next(reference_model.parameters()).device)
print("our model device:", next(model.parameters()).device)
print("input device:", idx_device.device)

with torch.no_grad():
    reference_logits = reference_model(idx_cpu).logits
    our_logits, _ = model(idx_device)

our_logits_cpu = our_logits.cpu()

print("reference:", reference_logits.shape)
print("ours     :", our_logits_cpu.shape)

diff = (reference_logits - our_logits_cpu).abs()

print("max diff :", diff.max().item())
print("mean diff:", diff.mean().item())

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

reference device: cpu
our model device: mps:0
input device: mps:0
reference: torch.Size([1, 10, 50257])
ours     : torch.Size([1, 10, 50257])
max diff : 0.07487106323242188
mean diff: 0.027617916464805603


In [44]:
print("reference:", reference_logits.shape)
print("ours     :", our_logits.shape)

reference: torch.Size([1, 10, 50257])
ours     : torch.Size([1, 10, 50257])


In [46]:
max_diff = (
    reference_logits - our_logits_cpu
).abs().max()

print("max absolute difference:", max_diff.item())

max absolute difference: 0.07487106323242188


In [47]:
mean_diff = (
    reference_logits - our_logits_cpu
).abs().mean()

print("mean absolute difference:", mean_diff.item())

mean absolute difference: 0.027617916464805603


In [48]:
reference_next = torch.argmax(
    reference_logits[:, -1, :],
    dim=-1,
)

our_next = torch.argmax(
    our_logits[:, -1, :],
    dim=-1,
)

print("reference token:", reference_next.item())
print("our token      :", our_next.item())

print(
    "reference:",
    tokenizer.decode(reference_next.tolist())
)

print(
    "ours     :",
    tokenizer.decode(our_next.tolist())
)

reference token: 198
our token      : 198
reference: 

ours     : 



In [49]:
k = 10

reference_top = torch.topk(
    reference_logits[:, -1, :],
    k=k,
    dim=-1,
)

our_top = torch.topk(
    our_logits[:, -1, :],
    k=k,
    dim=-1,
)

print("Reference:")
for token_id, score in zip(
    reference_top.indices[0],
    reference_top.values[0],
):
    print(
        repr(tokenizer.decode([token_id.item()])),
        score.item(),
    )

print("\nOurs:")
for token_id, score in zip(
    our_top.indices[0],
    our_top.values[0],
):
    print(
        repr(tokenizer.decode([token_id.item()])),
        score.item(),
    )

Reference:
'\n' -96.44969177246094
' The' -97.13166046142578
' "' -97.19883728027344
' He' -97.56451416015625
' It' -98.29246520996094
' She' -98.67424774169922
' They' -98.8168716430664
'\n\n' -99.4181137084961
' A' -99.68096160888672
' His' -99.73086547851562

Ours:
'\n' -96.43800354003906
' The' -97.1239242553711
' "' -97.18529510498047
' He' -97.5519027709961
' It' -98.28145599365234
' She' -98.66194915771484
' They' -98.80889892578125
'\n\n' -99.40609741210938
' A' -99.6703109741211
' His' -99.7184066772461


In [50]:
print(config.activation_function)

gelu_new


In [53]:
text = "The future of artificial intelligence"

idx = torch.tensor(
    [tokenizer.encode(text)],
    dtype=torch.long,
    device=device,
)

print("model device:", next(model.parameters()).device)
print("input device:", idx.device)

with torch.no_grad():
    hf_logits = reference_model(idx.cpu()).logits
    our_logits, loss = model(idx)

our_logits_cpu = our_logits.cpu()

print(
    "max diff:",
    (hf_logits - our_logits_cpu).abs().max().item()
)

print(
    "mean diff:",
    (hf_logits - our_logits_cpu).abs().mean().item()
)

hf_next = torch.argmax(hf_logits[:, -1, :], dim=-1)
our_next = torch.argmax(our_logits[:, -1, :], dim=-1)

print("HF :", repr(tokenizer.decode(hf_next.tolist())))
print("Ours:", repr(tokenizer.decode(our_next.tolist())))

model device: mps:0
input device: mps:0
max diff: 0.07472991943359375
mean diff: 0.030791040509939194
HF : ' is'
Ours: ' is'


In [55]:
prompt = "The future of artificial intelligence"

tokens = tokenizer.encode(prompt)

idx = torch.tensor(
    [tokens],
    dtype=torch.long,
    device = device
)

generated = model.generate(
    idx,
    max_new_tokens=50,
)

text = tokenizer.decode(
    generated[0].tolist()
)

print(text)

The future of artificial intelligence.

And what would happen to the human race if they were to turn their attention directly to the future of artificial intelligence? Would they be more likely to make intelligent decisions about what to choose instead of what it's learned and will they spend their


In [56]:
#After adding topK
prompt = "The future of artificial intelligence"
torch.manual_seed(42) #generated text should remain equivalent in behavior to the CPU version
idx = torch.tensor(
    [tokenizer.encode(prompt)],
    dtype=torch.long,
    device=device,
)

generated = model.generate(
    idx,
    max_new_tokens=50,
    top_k=50,
)

print(
    tokenizer.decode(
        generated[0].tolist()
    )
)

The future of artificial intelligence has been brought to my attention in recent years. Since the dawn of AI in a wide variety of fields and technologies, there has been considerable progress in understanding the human condition. This includes understanding the health risks related to aging. The potential to gain insights


In [57]:
print("device:", device)
print("model:", next(model.parameters()).device)
print("input:", idx.device)

device: mps
model: mps:0
input: mps:0


In [58]:
def get_batch(tokens, batch_size, block_size, device):
    max_start = len(tokens) - block_size - 1

    ix = torch.randint(
        max_start,
        (batch_size,),
    )

    x = torch.stack([
        tokens[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        tokens[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [59]:
text = """
The future of artificial intelligence is uncertain.
Large language models learn statistical patterns from text.
Transformers use attention to model relationships between tokens.
"""

encoded = tokenizer.encode(text)

tokens = torch.tensor(
    encoded,
    dtype=torch.long,
)

In [60]:
x, y = get_batch(
    tokens,
    batch_size=4,
    block_size=16,
    device=device,
)

In [61]:
print(x.shape)
print(y.shape)

torch.Size([4, 16])
torch.Size([4, 16])


In [63]:
logits, loss = model(x)

print(logits.shape) #At each of 4 x 16 posn it produces 50257 scores. So total logit values = 4×16×50257

torch.Size([4, 16, 50257])


In [64]:
print(tokenizer.decode(x[0].tolist()))
print(tokenizer.decode(y[0].tolist()))

.
Large language models learn statistical patterns from text.
Transformers use attention

Large language models learn statistical patterns from text.
Transformers use attention to


In [65]:
logits, loss = model(x, y)

print("x:", x.shape)
print("y:", y.shape)
print("logits:", logits.shape)
print("loss:", loss.item())

x: torch.Size([4, 16])
y: torch.Size([4, 16])
logits: torch.Size([4, 16, 50257])
loss: 6.256852626800537


In [66]:
model = GPT2(
    vocab_size=config.vocab_size,
    block_size=config.n_positions,
    n_layer=config.n_layer,
    n_head=config.n_head,
    n_embd=config.n_embd,
)

load_gpt2_weights(model, reference_model)

model = model.to(device)
model.eval()

idx = idx.cpu().to(device)

print("model device:", next(model.parameters()).device)
print("idx device:", idx.device)

with torch.no_grad():
    our_logits, _ = model(idx)

print("ours:", our_logits.shape)

model device: mps:0
idx device: mps:0
ours: torch.Size([1, 5, 50257])


In [70]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

model.train()

for step in range(50):
    optimizer.zero_grad(set_to_none=True)

    logits, loss = model(x, y)

    loss.backward()

    optimizer.step()

    if step % 5 == 0:
        print(f"step {step:02d} | loss {loss.item():.4f}")

step 00 | loss 6.2569
step 05 | loss 0.4537
step 10 | loss 0.0144
step 15 | loss 0.0050
step 20 | loss 0.0016
step 25 | loss 0.0003
step 30 | loss 0.0002
step 35 | loss 0.0002
step 40 | loss 0.0001
step 45 | loss 0.0001


In [71]:
model.eval()

with torch.no_grad():
    logits, loss = model(x, y)

print("final loss:", loss.item())

final loss: 8.968489419203252e-05


In [ ]:
'''This overfits the tiny batch successfully and shows that
forward pass works,
causal attention works,
gradients flow through the entire transformer,
parameters are actually being updated,
AdamW is working and loss implementation is correct enough to train, so,
the model can memorize a tiny batch :)'''

In [72]:
class DataLoaderLite:
    def __init__(self, B, T, tokenizer, text):
        self.B = B
        self.T = T
        self.tokenizer = tokenizer

        self.tokens = torch.tensor(
            tokenizer.encode(text),
            dtype=torch.long,
        )

        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T

        buf = self.tokens[
            self.current_position:
            self.current_position + B * T + 1
        ]

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        self.current_position += B * T

        if self.current_position + B * T + 1 > len(self.tokens):
            self.current_position = 0

        return x.to(device), y.to(device)

In [73]:
train_text = """
The future of artificial intelligence is uncertain.
Artificial intelligence systems learn patterns from data.
Language models predict the next token given previous tokens.
Transformers use attention to process sequences of tokens.
Deep learning models improve through optimization.
""" * 100

In [74]:
B = 4
T = 16

train_loader = DataLoaderLite(
    B=B,
    T=T,
    tokenizer=tokenizer,
    text=train_text,
)

x, y = train_loader.next_batch() #dw about working since we are not going to pass complete sequence at once

print("x:", x.shape)
print("y:", y.shape)
print("x:")
print(x)

print("y:")
print(y)

Token indices sequence length is longer than the specified maximum sequence length for this model (5000 > 1024). Running this sequence through the model will result in indexing errors


x: torch.Size([4, 16])
y: torch.Size([4, 16])
x:
tensor([[  198,   464,  2003,   286, 11666,  4430,   318,  8627,    13,   198,
          8001,  9542,  4430,  3341,  2193,  7572],
        [  422,  1366,    13,   198, 32065,  4981,  4331,   262,  1306, 11241,
          1813,  2180, 16326,    13,   198, 41762],
        [  364,   779,  3241,   284,  1429, 16311,   286, 16326,    13,   198,
         29744,  4673,  4981,  2987,   832, 23989],
        [   13,   198,   198,   464,  2003,   286, 11666,  4430,   318,  8627,
            13,   198,  8001,  9542,  4430,  3341]], device='mps:0')
y:
tensor([[  464,  2003,   286, 11666,  4430,   318,  8627,    13,   198,  8001,
          9542,  4430,  3341,  2193,  7572,   422],
        [ 1366,    13,   198, 32065,  4981,  4331,   262,  1306, 11241,  1813,
          2180, 16326,    13,   198, 41762,   364],
        [  779,  3241,   284,  1429, 16311,   286, 16326,    13,   198, 29744,
          4673,  4981,  2987,   832, 23989,    13],
        [  198

In [75]:
print(torch.equal(x[:, 1:], y[:, :-1]))

True


In [76]:
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

for step in range(20):
    x, y = train_loader.next_batch() #instead of roll using next_batch func

    optimizer.zero_grad(set_to_none=True)

    logits, loss = model(x, y)

    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(f"step {step:02d} | loss {loss.item():.4f}")

step 00 | loss 8.6422
step 05 | loss 1.8727
step 10 | loss 0.6703
step 15 | loss 0.3763


# Out of the mess Implementation

### Imports Configs Device

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import GPT2LMHeadModel, GPT2TokenizerFast

torch.manual_seed(1337)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()

reference_model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2"
)
reference_model.eval()

config = reference_model.config

print("device:", device)
print(f"parameters: {sum(p.numel() for p in reference_model.parameters()):,}")
print(f"parameters: {sum(p.numel() for p in reference_model.parameters()) / 1e6:.2f}M")

### Model

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5,
        )


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head

        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(block_size, block_size))

        self.register_buffer(
            "bias",
            mask.view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        att = att.masked_fill(
            self.bias[:, :, :T, :T] == 0,
            float("-inf"),
        )

        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v

        y = y.transpose(1, 2).contiguous()
        y = y.view(B, T, C)

        y = self.c_proj(y)
        y = self.resid_dropout(y)

        return y


class GPT2MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()

        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class GPT2Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()

        self.ln_1 = LayerNorm(n_embd)
        self.attn = CausalSelfAttention(
            n_embd,
            n_head,
            block_size,
            dropout,
        )

        self.ln_2 = LayerNorm(n_embd)
        self.mlp = GPT2MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        n_layer,
        n_head,
        n_embd,
        dropout=0.0,
    ):
        super().__init__()

        self.block_size = block_size

        self.transformer = nn.ModuleDict({
            "wte": nn.Embedding(vocab_size, n_embd),
            "wpe": nn.Embedding(block_size, n_embd),
            "h": nn.ModuleList([
                GPT2Block(
                    n_embd=n_embd,
                    n_head=n_head,
                    block_size=block_size,
                    dropout=dropout,
                )
                for _ in range(n_layer)
            ]),
            "ln_f": LayerNorm(n_embd),
        })

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.transformer["wte"].weight

    def forward(self, idx, targets=None):
        B, T = idx.shape

        assert T <= self.block_size, (
            f"sequence length {T} exceeds block size {self.block_size}"
        )

        tok_emb = self.transformer["wte"](idx)

        pos = torch.arange(
            T,
            device=idx.device,
        )

        pos_emb = self.transformer["wpe"](pos)

        x = tok_emb + pos_emb

        for block in self.transformer["h"]:
            x = block(x)

        x = self.transformer["ln_f"](x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            topk_probs, topk_indices = torch.topk(
                probs,
                top_k,
                dim=-1,
            )

            ix = torch.multinomial(
                topk_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                topk_indices,
                -1,
                ix,
            )

            idx = torch.cat(
                (idx, next_token),
                dim=1,
            )

        return idx

### Loading GPT-2 Weights

In [ ]:
def load_gpt2_weights(model, reference_model):
    with torch.no_grad():
        model.transformer["wte"].weight.copy_(
            reference_model.transformer.wte.weight
        )

        model.transformer["wpe"].weight.copy_(
            reference_model.transformer.wpe.weight
        )

        for i, block in enumerate(model.transformer["h"]):
            ref_block = reference_model.transformer.h[i]

            block.ln_1.weight.copy_(ref_block.ln_1.weight)
            block.ln_1.bias.copy_(ref_block.ln_1.bias)

            block.attn.c_attn.weight.copy_(
                ref_block.attn.c_attn.weight.T
            )
            block.attn.c_attn.bias.copy_(
                ref_block.attn.c_attn.bias
            )

            block.attn.c_proj.weight.copy_(
                ref_block.attn.c_proj.weight.T
            )
            block.attn.c_proj.bias.copy_(
                ref_block.attn.c_proj.bias
            )

            block.ln_2.weight.copy_(ref_block.ln_2.weight)
            block.ln_2.bias.copy_(ref_block.ln_2.bias)

            block.mlp.c_fc.weight.copy_(
                ref_block.mlp.c_fc.weight.T
            )
            block.mlp.c_fc.bias.copy_(
                ref_block.mlp.c_fc.bias
            )

            block.mlp.c_proj.weight.copy_(
                ref_block.mlp.c_proj.weight.T
            )
            block.mlp.c_proj.bias.copy_(
                ref_block.mlp.c_proj.bias
            )

        model.transformer["ln_f"].weight.copy_(
            reference_model.transformer.ln_f.weight
        )
        model.transformer["ln_f"].bias.copy_(
            reference_model.transformer.ln_f.bias
        )

    return model

### Constructing Tokenizer and Model

In [ ]:
model = GPT2(
    vocab_size=config.vocab_size,
    block_size=config.n_positions,
    n_layer=config.n_layer,
    n_head=config.n_head,
    n_embd=config.n_embd,
)

load_gpt2_weights(model, reference_model)

model = model.to(device)
model.eval()

tokenizer = GPT2TokenizerFast.from_pretrained(
    "openai-community/gpt2"
)

print("model device:", next(model.parameters()).device)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

### Verification

In [ ]:
text = "The future of artificial intelligence"

idx = torch.tensor(
    [tokenizer.encode(text)],
    dtype=torch.long,
    device=device,
)

with torch.no_grad():
    hf_logits = reference_model(idx.cpu()).logits
    our_logits, loss = model(idx)

our_logits_cpu = our_logits.cpu()

print(
    "max diff:",
    (hf_logits - our_logits_cpu).abs().max().item(),
)

print(
    "mean diff:",
    (hf_logits - our_logits_cpu).abs().mean().item(),
)

hf_next = torch.argmax(
    hf_logits[:, -1, :],
    dim=-1,
)

our_next = torch.argmax(
    our_logits[:, -1, :],
    dim=-1,
)

print("HF :", repr(tokenizer.decode(hf_next.tolist())))
print("Ours:", repr(tokenizer.decode(our_next.tolist())))

### current loss sanity check

In [ ]:
text = "The future of artificial intelligence is uncertain. "

tokens = tokenizer.encode(text)

tokens = torch.tensor(
    tokens,
    dtype=torch.long,
)

block_size = 16
batch_size = 4

x = tokens[:block_size].unsqueeze(0).repeat(batch_size, 1).to(device)
y = torch.roll(x, shifts=-1, dims=1)

with torch.no_grad():
    logits, loss = model(x, y)

print("x:", x.shape)
print("y:", y.shape)
print("logits:", logits.shape)
print("loss:", loss.item())

### Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

model.train()

for step in range(50):
    optimizer.zero_grad(set_to_none=True)

    logits, loss = model(x, y)

    loss.backward()

    optimizer.step()

    if step % 5 == 0:
        print(f"step {step:02d} | loss {loss.item():.4f}")

In [ ]:
model.eval()

with torch.no_grad():
    logits, loss = model(x, y)

print("final loss:", loss.item())

### DataLoader

In [ ]:
class DataLoaderLite:
    def __init__(self, B, T, tokenizer, text):
        self.B = B
        self.T = T
        self.tokenizer = tokenizer

        self.tokens = torch.tensor(
            tokenizer.encode(text),
            dtype=torch.long,
        )

        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T

        buf = self.tokens[
            self.current_position:
            self.current_position + B * T + 1
        ]

        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)

        self.current_position += B * T

        if self.current_position + B * T + 1 > len(self.tokens):
            self.current_position = 0

        return x.to(device), y.to(device)